In [0]:
use catalog tpcds_lakehouse;

In [0]:
-- Aggregate FIRST, then join — reduces shuffle size massively
WITH date_keys AS (
    SELECT d_date_sk, d_year
    FROM silver.date_dim
    WHERE d_year BETWEEN 1998 AND 2002   -- push filter early
),

store_agg AS (
    SELECT ss_sold_date_sk AS date_sk,
           SUM(ss_net_paid) AS store_net_paid
    FROM silver.store_sales
    WHERE ss_sold_date_sk IN (SELECT d_date_sk FROM date_keys)
    GROUP BY ss_sold_date_sk              -- aggregate before join
),

web_agg AS (
    SELECT ws_sold_date_sk AS date_sk,
           SUM(ws_net_paid) AS web_net_paid
    FROM silver.web_sales
    WHERE ws_sold_date_sk IN (SELECT d_date_sk FROM date_keys)
    GROUP BY ws_sold_date_sk
),

catalog_agg AS (
    SELECT cs_sold_date_sk AS date_sk,
           SUM(cs_net_paid) AS catalog_net_paid
    FROM silver.catalog_sales
    WHERE cs_sold_date_sk IN (SELECT d_date_sk FROM date_keys)
    GROUP BY cs_sold_date_sk
)

SELECT
    d.d_year,
    ROUND(COALESCE(SUM(s.store_net_paid),   0) / 1e6, 2) AS store_net_paid_M,
    ROUND(COALESCE(SUM(w.web_net_paid),     0) / 1e6, 2) AS web_net_paid_M,
    ROUND(COALESCE(SUM(c.catalog_net_paid), 0) / 1e6, 2) AS catalog_net_paid_M,
    ROUND(COALESCE(SUM(s.store_net_paid),   0) / 1e6 +
          COALESCE(SUM(w.web_net_paid),     0) / 1e6 +
          COALESCE(SUM(c.catalog_net_paid), 0) / 1e6, 2) AS total_net_paid_M
FROM date_keys d
LEFT JOIN store_agg   s ON s.date_sk = d.d_date_sk
LEFT JOIN web_agg     w ON w.date_sk = d.d_date_sk
LEFT JOIN catalog_agg c ON c.date_sk = d.d_date_sk
GROUP BY d.d_year
ORDER BY d.d_year;